# Week 4a.1: LangGraph Basics

We've seen how to build an agent with `create_agent`, which runs the whole propose-execute-return loop for us. Today we'll take a closer look at how it works. 

`create_agent` is built on [**LangGraph**](https://docs.langchain.com/oss/python/langgraph/graph-api), a library for describing agents as graphs. If we understand the graph layer, we can build flows that `create_agent` cannot give us such as branching logic and state management.

We'll start by building a first graph, a sequential chain, and a router.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
# run in terminal: uv pip install langchain_openrouter

from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="google/gemma-4-31b-it:free", max_retries=4)

In [ ]:
# run in terminal: uv pip install langchain_nvidia_ai_endpoints
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b", reasoning=False)
#model = ChatNVIDIA(model="meta/muse-glimmer-30b")

In [ ]:
model.invoke("What's an embedding in one sentence")

## 1. A Simple Graph: The three core elements

Every LangGraph application is built from three things:

- **State**: a shared data structure that every step can read and update. We define its shape with a `TypedDict`.
- **Nodes**: plain Python functions. A node receives the current state and returns a dictionary containing only the keys it wants to update.
- **Edges**: connections that decide which node runs next. `START` and `END` mark where execution enters and leaves the graph.

Docs: https://docs.langchain.com/oss/python/langgraph/graph-api

In [ ]:
from typing import Literal, TypedDict

class State(TypedDict):
    message: str

A node is an ordinary function. It takes the state and returns the update it wants to make.

In [ ]:
def add_greeting(state: State) -> dict:
    return {"message": "Hello, " + state["message"]}

In [ ]:
from langgraph.graph import StateGraph, START, END

# TODO
# construct the state graph 

# add the node "add_greeting"

# add the edges: START -> add_greeting -> END

# compile the StateGraph


In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

### Invoking a Graph
Runs the graph with an input by sending a state message through the entire graph

`invoke` takes a dictionary shaped like the state and returns the final state. 

Each node's return value overwrites the specified state element(s)

In [ ]:
graph.invoke({"message":"Agentic AI class"})

<hr />

## 2. A sequential graph
Let's now add multiple nodes sequentially.

Construction always involves the same steps:
 - Instantiate a builder around the state
 - Add nodes
 - Add edges
 - Compile.

This is the graph we are about to build:

<img src="figures/graph_sequential.png" width="230" alt="sequential graph">

In [ ]:
def add_greeting(state: State) -> dict:
    return {"message": "Hello, " + state["message"]}

def shout(state: State) -> dict:
    return {"message": state["message"].upper() + "!"}

In [ ]:
from langgraph.graph import StateGraph, START, END

# construct the state graph

# add nodes

# add edges START -> add_greeting -> shout -> END


In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"message": "agentic AI class"})

In [ ]:
for chunk in graph.stream({"message": "agentic AI class"}, stream_mode="values"):
    print("Stream step state:", chunk)

**Notice there is no LLM anywhere yet. A graph is just orchestration; what runs inside each node is up to us.**

## 3. A chain

We'll now put a model inside the nodes. This is the **prompt chaining** pattern from the Week 1 reading (Building Effective Agents): each step's output becomes the next step's input, and the route through the graph is fixed.

<img src="figures/graph_chain.png" width="230" alt="sequential chain">

In [ ]:
class JokeState(TypedDict):
    topic: str
    joke: str
    improved_joke: str

In [ ]:
def write_joke(state: JokeState) -> dict:
    response = model.invoke(f"Write a short, clean joke about {state['topic']}.")
    return {"joke": response.text}

In [ ]:
chain_builder = StateGraph(JokeState)

# add node: write_joke
chain_builder.add_node("write_joke", write_joke)

# add edges: START -> write_joke -> improve_joke -> END
chain_builder.add_edge(START, "write_joke")
chain_builder.add_edge("write_joke", END)

chain = chain_builder.compile()

display(Image(chain.get_graph().draw_mermaid_png()))

result = chain.invoke({"topic": "college professors"})
print(result)

In [ ]:
print(result["joke"])

One practical detail: we need to store `response.text`, not `response.content`. 

Some providers (Gemini included) return `content` as a list of content blocks rather than a plain string; `.text` always gives you the reply as a string.

In [ ]:
#TODO: write the second node, improve_joke.
# It should ask the model to make state['joke'] funnier by adding a follow-up line,
# and return the result under the key 'improved_joke'.


In [ ]:
chain_builder = StateGraph(JokeState)

# add nodes: write_joke and improve_joke
chain_builder.add_node("write_joke", write_joke)
chain_builder.add_node("improve_joke", improve_joke)

# add edges: START -> write_joke -> improve_joke -> END
chain_builder.add_edge(START, "write_joke")
chain_builder.add_edge("write_joke", "improve_joke")
chain_builder.add_edge("improve_joke", END)

chain = chain_builder.compile()
display(Image(chain.get_graph().draw_mermaid_png()))

In [ ]:
result = chain.invoke({"topic": "college professors"})

print(result["joke"])
print("---")
print(result["improved_joke"])

In [ ]:
for chunk in chain.stream({"topic": "college professors"}, stream_mode="values"):
    print("Stream step state:", chunk)

The model did the writing, but it made no decisions about control flow. The route was fixed when we drew the edges. That makes this a **workflow**, not an agent.

## 4. A router

So far every edge was fixed, with no branching/decision making. 

A **conditional edge** is used to make branching decisions. A conditional edge needs a function that inspects the state and returns the name of the node to run next.

The example: a support desk that triages incoming messages to a billing team or a general queue.

In these diagrams, solid arrows are fixed edges; **dashed red arrows are conditional edges**, where the next node is decided at runtime:

<img src="figures/graph_router.png" width="340" alt="router graph">

In [ ]:
class SupportState(TypedDict):
    message: str
    reply: str

def billing(state: SupportState) -> dict:
    return {"reply": "Billing team: we will review the charge on your account."}

def general(state: SupportState) -> dict:
    return {"reply": "Support: thanks for reaching out, how can we help?"}

In [ ]:
# the router function needs to return a node name
def route(state: SupportState) -> Literal["billing", "general"]:
    text = state["message"].lower()
    if any(word in text for word in ["refund", "charge", "billing", "payment"]):
        return "billing"
    return "general"

In [ ]:
support_messages = [SupportState(message="I want a refund for my order"),
           SupportState(message="Do you ship to Canada?")]

In [ ]:
route(support_messages[1])

In [ ]:
router_builder = StateGraph(SupportState)

router_builder.add_node("billing", billing)
router_builder.add_node("general", general)

# TODO: add the conditional edge from the start node to route

router_builder.add_edge("billing", END)
router_builder.add_edge("general", END)

router = router_builder.compile()
display(Image(router.get_graph().draw_mermaid_png()))

### Adding a Fake Router Node
Not really a node because it doesn't process the state
- To the router node, add a lambda function that retuns the state as is: lambda state:state

In [ ]:
router_builder = StateGraph(SupportState)

router_builder.add_node("billing", billing)
router_builder.add_node("general", general)
# TODO: add the fake add node with lambda function

# TODO: add the edge START -> router and conditional edge
router_builder.add_edge("billing", END)
router_builder.add_edge("general", END)

router = router_builder.compile()
display(Image(router.get_graph().draw_mermaid_png()))

In [ ]:
print(router.invoke(support_messages[0])["reply"])

In [ ]:
print(router.invoke(support_messages[1])["reply"])

The routing logic is hard-coded keywords, and keywords are brittle. "My card was hit twice for one order" is clearly a billing problem, but none of our keywords appear in it. Let the model classify instead.

In [ ]:
def llm_route(state: SupportState) -> Literal["billing", "general"]:
    response = model.invoke(
        "Classify this customer message as billing or general. "
        "Answer with exactly one word: billing or general.\n\n"
        f"Message: {state['message']}"
    )
    answer = response.text.strip().lower()
    return "billing" if "billing" in answer else "general"

In [ ]:
llm_route(support_messages[1])

In [ ]:
llm_router_builder = StateGraph(SupportState)

llm_router_builder.add_node("billing", billing)
llm_router_builder.add_node("general", general)
llm_router_builder.add_node("router", lambda state:state)

llm_router_builder.add_edge(START, "router")
llm_router_builder.add_conditional_edges("router", llm_route)
llm_router_builder.add_edge("billing", END)
llm_router_builder.add_edge("general", END)

llm_router = llm_router_builder.compile()

display(Image(llm_router.get_graph().draw_mermaid_png()))


In [ ]:
print(llm_router.invoke({"message": "I have two transactions for the same order." }))

In [ ]:
print(llm_router.invoke({"message": "How can I return my order?" })["reply"])

In [ ]:
for chunk in llm_router.stream(support_messages[0], stream_mode="values"):
    print("Stream step state:", chunk)

An agent needs more: decide, act, look at the result, decide again. That takes a loop in the graph, and the model's **tool calls** become the routing signal. That is what we'll cover next.